# 02 · RAG Foundations

**Goal:** understand Retrieval-Augmented Generation without any external vector
DB. We simulate `pgvector`-style retrieval in pure Python: embed a small corpus,
match a query against it, and **stuff the top results into the prompt** so the
model answers from grounded facts instead of hallucinating.

## 1. A tiny knowledge corpus

Imagine these are rows in a `pgvector` table — each a chunk of a "role
requirements" knowledge base.

In [1]:
CORPUS = [
    "A Machine Learning Engineer must be fluent in Python and PyTorch or TensorFlow.",
    "MLOps skills include Docker, CI/CD, and model monitoring with tools like MLflow.",
    "Data version control (DVC) and experiment tracking are expected for senior ML roles.",
    "Frontend engineers focus on React, TypeScript, and accessibility (a11y).",
    "Product managers write PRDs and prioritize with frameworks like RICE.",
    "Vector databases (pgvector, Pinecone) power retrieval for RAG systems.",
]
print(f"{len(CORPUS)} documents in corpus.")

6 documents in corpus.


## 2. A toy embedding + cosine similarity

Real systems call an embedding model. To stay dependency-free and deterministic,
we use a bag-of-words vector. The **retrieval logic is identical** — only the
vectorizer changes when you swap in real embeddings from `pgvector`.

In [2]:
import math
import re
from collections import Counter

def tokenize(text: str):
    return re.findall(r"[a-z0-9]+", text.lower())

def embed(text: str) -> Counter:
    return Counter(tokenize(text))

def cosine(a: Counter, b: Counter) -> float:
    common = set(a) & set(b)
    dot = sum(a[t] * b[t] for t in common)
    na = math.sqrt(sum(v * v for v in a.values()))
    nb = math.sqrt(sum(v * v for v in b.values()))
    return dot / (na * nb) if na and nb else 0.0

CORPUS_VECS = [embed(doc) for doc in CORPUS]

## 3. Retrieve the top-k most relevant chunks

In [3]:
def retrieve(query: str, k: int = 3):
    qv = embed(query)
    scored = sorted(
        ((cosine(qv, dv), doc) for dv, doc in zip(CORPUS_VECS, CORPUS)),
        key=lambda x: x[0],
        reverse=True,
    )
    return [doc for score, doc in scored[:k] if score > 0]

query = "What does a machine learning engineer need for MLOps?"
hits = retrieve(query)
for h in hits:
    print("•", h)

• A Machine Learning Engineer must be fluent in Python and PyTorch or TensorFlow.
• Vector databases (pgvector, Pinecone) power retrieval for RAG systems.
• MLOps skills include Docker, CI/CD, and model monitoring with tools like MLflow.


## 4. Ground the prompt (prevent hallucination)

We inject retrieved chunks as **context** and instruct the model to answer *only*
from that context. This is the core RAG move.

In [4]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()
client = Groq(api_key=os.environ["GROQ_API_KEY"])

def answer_grounded(query: str) -> str:
    context = "\n".join(f"- {c}" for c in retrieve(query))
    system = (
        "Answer the question using ONLY the provided context. "
        "If the context is insufficient, say 'I don't have enough information.' "
        "Never invent facts."
    )
    user = f"CONTEXT:\n{context}\n\nQUESTION: {query}"
    r = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": user}],
        temperature=0.0,
    )
    return r.choices[0].message.content

print(answer_grounded(query))

A Machine Learning Engineer needs Docker, CI/CD, and model monitoring with tools like MLflow for MLOps.


## 5. Prove grounding works: ask something out-of-corpus

A well-grounded model should **refuse** rather than hallucinate when the answer
isn't in the retrieved context.

In [5]:
print(answer_grounded("What is the capital of France?"))

Paris.


### Recap
- RAG = retrieve relevant chunks, then stuff them into the prompt as context.
- Retrieval = vectorize query + docs, rank by cosine similarity (here bag-of-words;
  in production, embeddings in `pgvector`).
- Instructing the model to answer **only from context** curbs hallucination.

In CareerForge, the same grounding idea keeps the gap-analysis agent tied to the
actual resume and job description rather than generic advice.

Next: **03_agentic_workflows** — state, routing, and tools.